# NeMo-Skills Competitive Programmer Finetuning

This notebook prepares the OpenCodeReasoning-2 dataset (CPP split) and finetunes a Qwen2.5-Coder-7B-Instruct model. Storage management is optimized to handle large base datasets.

In [21]:
%pip install git+https://github.com/NVIDIA-NeMo/Skills.git
%pip install datasets==2.16.0

  Cloning https://github.com/NVIDIA-NeMo/Skills.git to /tmp/pip-req-build-l7bcubj8
  Running command git clone --filter=blob:none --quiet https://github.com/NVIDIA-NeMo/Skills.git /tmp/pip-req-build-l7bcubj8
  Resolved https://github.com/NVIDIA-NeMo/Skills.git to commit acec1b0c6e7614c5a35095e8e192be89b0fa02f7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/NVIDIA/compute-eval.git (to revision e01a5d2) to /tmp/pip-install-7k_qk1rf/compute-eval_1cc3a764ca06446093a7292490ca2684
  Running command git clone --filter=blob:none --quiet https://github.com/NVIDIA/compute-eval.git /tmp/pip-install-7k_qk1rf/compute-eval_1cc3a764ca06446093a7292490ca2684
  Running command git checkout -q e01a5d2
  Resolved https://github.com/NVIDIA/compute-eval.git to commit e01a5d2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject

In [ ]:
!ns setup
# /content:/workspace, /content/hf_cache:/hf_models
# /hf_models
# PYTHONPATH=/nemo_run/code

In [ ]:
import os
from huggingface_hub import snapshot_download
os.environ["NEMO_SKILLS_CONFIGS"] = "/content/cluster_configs"
os.environ["PYTHONPATH"] = "/content/Skills"
os.environ["NVIDIA_API_KEY"] = "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"
os.environ["HF_TOKEN"] = "xxx"  # huggingface.co/settings/tokens
MY_TOKEN = "xxx"

In [19]:
%cd /content/Skills
!git pull

/content/Skills
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 1.12 KiB | 383.00 KiB/s, done.
From https://github.com/Belaleatsbanana/Skills
   390aee77..0a462afe  main       -> origin/main
Updating 390aee77..0a462afe
Fast-forward
 nemo_notebook.ipynb                                | 42 ++++++++++++----------
 .../opencodereasoning/scripts/prepare_questions.py |  6 +++-
 2 files changed, 28 insertions(+), 20 deletions(-)


In [3]:
import datasets
print(datasets.__version__)

4.8.4


In [3]:
%cd /content
!git clone https://github.com/Belaleatsbanana/Skills.git
%cd /content/Skills

/content
Cloning into 'Skills'...
remote: Enumerating objects: 40778, done.
remote: Counting objects: 100% (1887/1887), done.
remote: Compressing objects: 100% (860/860), done.
remote: Total 40778 (delta 1549), reused 1070 (delta 1024), pack-reused 38891 (from 4)
Receiving objects: 100% (40778/40778), 383.35 MiB | 53.15 MiB/s, done.
Resolving deltas: 100% (29538/29538), done.
/content/Skills


In [3]:
ls

cluster_configs/  hf_cache/  opencodereasoning/  sample_data/  Skills/


## Step 1: Data Preparation

Extract questions from the OCR2 dataset. Original solutions are intentionally skipped as we will generate higher quality reasoning traces using a strong model (DeepSeek-R1).

In [22]:
!HF_HUB_ENABLE_HF_TRANSFER=1 python /content/Skills/recipes/opencodereasoning/scripts/prepare_questions.py \
    --output_dir /content/opencodereasoning/data \
    --cache_dir /content/hf_cache

[STEP 1] Loading OpenCodeReasoning-2 'cpp' split...
Resolving data files: 100% 70/70 [00:00<00:00, 34383.57it/s]
Resolving data files: 100% 59/59 [00:00<00:00, 106757.52it/s]
Resolving data files: 100% 70/70 [00:00<00:00, 72067.08it/s]
Resolving data files: 100% 59/59 [00:00<00:00, 194853.49it/s]
[STEP 2] Sorting questions by dataset using Arrow (High Speed)...
[STEP 3] Starting extraction of 1174475 questions...
Processing Questions:   0% 0/1174475 [00:00<?, ?it/s]
[STEP] Fetching source dataset from HuggingFace: apps





Generating train split: 0 examples [00:00, ? examples/s]
Generating train split: 353 examples [00:00, 3501.34 examples/s]
Generating train split: 782 examples [00:00, 3961.17 examples/s]
Generating train split: 1693 examples [00:00, 2282.58 examples/s]
Generating train split: 2000 examples [00:00, 1840.60 examples/s]
Generating train split: 2356 examples [00:01, 1411.01 examples/s]
Generating train split: 3000 examples [00:01, 2066.96 examples/s]
Generating train sp

## Step 2: Generate Solutions

Generate synthetic solutions with DeepSeek-R1 via NVIDIA NIM to get high-quality reasoning traces.

In [8]:
# This uses the 'demo' mode which connects to NVIDIA NIM (requires NVIDIA_API_KEY environment variable)
!python /content/Skills/recipes/opencodereasoning/pipeline/prepare_solutions.py --mode demo

Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 45, in <module>
    PACKAGE_DISTRIBUTION_MAPPING = importlib.metadata.packages_distributions()
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/metadata/__init__.py", line 947, in packages_distributions
    for pkg in _top_level_declared(dist) or _top_level_inferred(dist):
                                            ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/metadata/__init__.py", line 959, in _top_level_inferred
    for f in always_iterable(dist.files)
                             

## Step 3: Prepare SFT Data

Apply formatting for finetuning using the filtered R1 outputs.

In [ ]:
# Pointing to the filtered output from the R1 generation pipeline
input_path = "/workspace/opencodereasoning/data/solution-sdg-toy/filtered/output.jsonl"
!python -m nemo_skills.training.prepare_data \
    ++input_files={input_path} \
    ++output_path=/workspace/opencodereasoning/sft-data.jsonl \
    ++prompt_config=generic/math \
    ++add_unlabeled=true

## Step 4: Finetune Qwen2.5-Coder-7B-Instruct

Train using NeMo-RL.

In [ ]:
!ns nemo_rl sft \
    --cluster=slurm \
    --training_data=/workspace/opencodereasoning/sft-data.jsonl \
    --hf_model=Qwen/Qwen2.5-Coder-7B-Instruct \
    --backend=megatron \
    --expname=qwen-coder-sft \
    ++policy.max_total_sequence_length=8192 \
    ++sft.max_num_epochs=2